# XiDepth v2.0 — Track 1: ResNet-18 Baseline, Parity Gate & Step Timing

This notebook executes the rigorous validation and baseline training workflow:
1. **Prerequisites & Verification**: Checks GPU and attaches `awsaf49/kitti-eigen-split`.
2. **Clone Audited Branch**: Clones `v2-eval-audit` branch.
3. **Gate 1 (Dataset Integrity)**: Runs `verify_dataset_paths.py` (Must exit code 0).
4. **Gate 2 (Parity Test)**: Evaluates official Monodepth2 weights on the 652 benchmark (Must hit AbsRel ≈ 0.090, exits 1 if outside tolerance).
5. **Step Timing (100 Steps)**: Measures exact dataloader + GPU latency to determine realistic epoch durations.
6. **Track 1 Baseline Training**: Trains Pretrained ResNet-18 baseline with `--resume` support across sessions.
7. **Eigen Benchmark Evaluation**: Evaluates trained baseline model on the 652 benchmark.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


In [ ]:
# 2. Locate KITTI dataset on Kaggle
import os, glob
input_dirs = glob.glob("/kaggle/input/*kitti*")
print("Found candidate input dirs:", input_dirs)
dataset_dir = input_dirs[0] if input_dirs else "/kaggle/input/kitti-eigen-split"
print(f"Using dataset directory: {dataset_dir}")


In [ ]:
# 3. Clone audited branch v2-eval-audit and install dependencies
!git clone -b v2-eval-audit https://github.com/ASMITSARKAR/XiDepth-Monocular-2.0.git xidepth_repo
%cd xidepth_repo
!pip install -r requirements.txt

# Ensure official weights (verified URL) and improved GT are present
!mkdir -p checkpoints/monodepth2_official data
!wget -nc --show-progress https://storage.googleapis.com/niantic-lon-static/research/monodepth2/mono_640x192.zip -O checkpoints/monodepth2_official/mono_640x192.zip
!unzip -o -q checkpoints/monodepth2_official/mono_640x192.zip -d checkpoints/monodepth2_official/
!wget -nc --show-progress "https://www.dropbox.com/scl/fi/dg7eskv5ztgdyp4ippqoa/gt_depths.npz?rlkey=qb39aajkbhmnod71rm32136ry&dl=1" -O data/gt_depths.npz

assert os.path.isfile("checkpoints/monodepth2_official/encoder.pth"), "Missing encoder.pth"
assert os.path.isfile("checkpoints/monodepth2_official/depth.pth"), "Missing depth.pth"
assert os.path.isfile("data/gt_depths.npz"), "Missing gt_depths.npz"
print("Weights and ground truth verified on disk.")


In [ ]:
# 4. Gate 1: Verify all dataset paths and calib files exist on disk (Must Exit 0)
!python scripts/verify_dataset_paths.py --dataset_dir "$dataset_dir"


In [ ]:
# 5. Gate 2: Official Monodepth2 Parity Benchmark Test (Exits 1 if outside tolerance 0.005 of 0.090)
!python scripts/eval.py \
    --model monodepth2_official \
    --checkpoint checkpoints/monodepth2_official \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --gate_tolerance 0.005 \
    --output_json /kaggle/working/monodepth2_parity_results.json


In [ ]:
# 6. Step Timing: Time exactly 100 training steps (1,200 samples @ batch 12)
import time
t0 = time.time()
!python scripts/train.py \
    --model resnet18 \
    --pretrained \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_timing \
    --batch_size 12 \
    --epochs 1 \
    --max_samples 1200 \
    --num_workers 4 \
    --use_amp \
    --log_freq 20
t_elapsed = time.time() - t0

s_per_step = t_elapsed / 100.0
steps_per_epoch = 39810 // 12  # 3,317 steps
epoch_hours = (steps_per_epoch * s_per_step) / 3600.0
max_safe_epochs = int(10.0 / epoch_hours)  # Allocate max 10 hours for 12h Kaggle cap

print("=" * 60)
print(f"100 Steps Elapsed Time: {t_elapsed:.1f} s ({s_per_step:.2f} s/step)")
print(f"Projected 1 Epoch Time: {epoch_hours:.2f} hours")
print(f"Safe Epochs within single 12h session (10h budget): {max_safe_epochs} epochs")
print(f"Total 20 Epochs require: {epoch_hours * 20:.1f} hours ({20 / max_safe_epochs:.1f} sessions using --resume)")
print("=" * 60)


In [ ]:
# 7. Launch Track 1 Baseline Training (Pretrained ResNet-18)
# Set --epochs to fit your session budget, and use --resume to continue across sessions
!python scripts/train.py \
    --model resnet18 \
    --pretrained \
    --dataset_dir "$dataset_dir" \
    --split_dir data/splits/eigen_zhou \
    --checkpoint_dir /kaggle/working/checkpoints_resnet18 \
    --batch_size 12 \
    --epochs 10 \
    --lr 1e-4 \
    --lr_step_size 15 \
    --num_workers 4 \
    --use_amp \
    --log_freq 100

# To resume in a subsequent session after uploading previous checkpoints:
# !python scripts/train.py \
#     --model resnet18 \
#     --pretrained \
#     --resume /kaggle/input/my-previous-run/checkpoint_epoch_10.pth \
#     --dataset_dir "$dataset_dir" \
#     --split_dir data/splits/eigen_zhou \
#     --checkpoint_dir /kaggle/working/checkpoints_resnet18 \
#     --batch_size 12 \
#     --epochs 20 \
#     --num_workers 4 \
#     --use_amp


In [ ]:
# 8. Evaluate Track 1 Pretrained Model on Eigen Benchmark (652 files)
!python scripts/eval.py \
    --model resnet18 \
    --checkpoint /kaggle/working/checkpoints_resnet18/best_model.pth \
    --dataset_dir "$dataset_dir" \
    --split_file data/splits/eigen_benchmark/test_files.txt \
    --gt_path data/gt_depths.npz \
    --output_json /kaggle/working/resnet18_pretrained_results.json
